In [ ]:
import pandas as pd
import numpy as np
import ast
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pickle
import os
from pathlib import Path

# Setup paths that work whether run from notebooks/ or project root
BASE_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_DIR = BASE_DIR / 'datasets'
MODEL_DIR = BASE_DIR / 'best_model'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# 1. Load datasets
movies = pd.read_csv(DATA_DIR / 'tmdb_5000_movies.csv')
credits = pd.read_csv(DATA_DIR / 'tmdb_5000_credits.csv')
movies = movies.merge(credits, on='title')

# 2. Select relevant features and handle missing values
movies = movies[['movie_id', 'title', 'overview', 'genres', 'keywords', 'cast', 'crew']]
movies = movies.dropna()

# Helper function to parse JSON-like columns
def convert(obj):
    return [i['name'] for i in ast.literal_eval(obj)]

movies['genres'] = movies['genres'].apply(convert)
movies['keywords'] = movies['keywords'].apply(convert)

def convert_cast(obj):
    cast_list = []
    for i in ast.literal_eval(obj)[:3]:
        cast_list.append(i['name'])
    return cast_list

def fetch_director(obj):
    for i in ast.literal_eval(obj):
        if i['job'] == 'Director':
            return [i['name']]
    return []

movies['cast'] = movies['cast'].apply(convert_cast)
movies['crew'] = movies['crew'].apply(fetch_director)
movies['overview'] = movies['overview'].apply(lambda x: x.split())

# Remove spaces between names (e.g., 'Sam Worthington' -> 'SamWorthington')
for col in ['genres', 'keywords', 'cast', 'crew']:
    movies[col] = movies[col].apply(lambda x: [i.replace(' ', '') for i in x])

# Create tags column
movies['tags'] = movies['overview'] + movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew']
new_df = movies[['movie_id', 'title', 'tags']].copy()
new_df['tags'] = new_df['tags'].apply(lambda x: ' '.join(x).lower())

# 3. Vectorization
cv = CountVectorizer(max_features=5000, stop_words='english')
vectors = cv.fit_transform(new_df['tags']).toarray()

# 4. Calculate similarity matrix
similarity = cosine_similarity(vectors)

# 5. Export pickle files
with open(MODEL_DIR / 'movie_dict.pkl', 'wb') as file:
    pickle.dump(new_df.to_dict(), file)

with open(MODEL_DIR / 'similarity.pkl', 'wb') as file:
    pickle.dump(similarity, file)

print('Pickle files generated successfully!')
print(f'Movies processed: {len(new_df)}')
print(f'Files saved in: {MODEL_DIR.resolve()}')
